# Jupyter Chat Components Demo

This notebook showcases the components provided by the
[`jupyter-chat-components`](https://github.com/jtpio/jupyter-chat-components) extension.

The extension registers a custom MIME renderer for the `application/vnd.jupyter.chat.components`
MIME type, which can be used to display rich, interactive UI components in notebook outputs.

## Programmatic Usage

Components are rendered via the Jupyter display protocol using the custom MIME type.

The display protocol uses two parts:

- **Data bundle**: `{MIME_TYPE: "<component-name>"}` — selects which component to render
- **Metadata**: `{MIME_TYPE: {...}}` — provides the component's props/configuration

This notebook uses the [JavaScript kernel](https://github.com/jupyterlite/javascript-kernel) and its
[`display()`](https://github.com/jupyterlite/javascript-kernel#rich-output) function
with the `_toMime()` method to send custom MIME bundles:

```javascript
const MIME_TYPE = "application/vnd.jupyter.chat.components";

display(
  // Data bundle: selects the component to render
  { _toMime: () => ({ [MIME_TYPE]: "tool-call" }) },
  // Metadata: provides the component's configuration
  {
    [MIME_TYPE]: {
      toolName: "my_tool",
      input: '{"key": "value"}',
      status: "completed",
    },
  },
);
```

## ToolCall Component

The `ToolCall` component renders tool execution information in a collapsible UI element,
showing the tool name, input, output, and status.

**Component name**: `"tool-call"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `toolName` | `string` | Yes | Name of the tool being called |
| `input` | `string` | Yes | Input data or parameters passed to the tool |
| `status` | `ToolCallStatus` | Yes | Current execution status (see below) |
| `summary` | `string` | No | Short summary displayed next to the tool name |
| `output` | `string` | No | Output or result from the tool execution |

### Status Types

| Status | Description |
|--------|-------------|
| `"pending"` | Tool is currently running |
| `"awaiting_approval"` | Tool requires user approval before executing |
| `"approved"` | User approved execution and the tool is running |
| `"rejected"` | User rejected execution |
| `"completed"` | Tool finished successfully |
| `"error"` | Tool encountered an error |

### Examples

The following helper function wraps `display()` for convenience:

In [ ]:
const MIME_TYPE = "application/vnd.jupyter.chat.components";

function showToolCall({ toolName, input, status = "completed", summary, output }) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "tool-call" }) },
    { [MIME_TYPE]: { toolName, input, status, summary, output } },
  );
}

#### Completed Tool Call

A tool call that has finished executing successfully:

In [ ]:
showToolCall({
  toolName: "list_files",
  input: '{"path": "/home/user/project"}',
  status: "completed",
  summary: "Listed 5 files",
  output: '["README.md", "setup.py", "main.py", "utils.py", "test_main.py"]',
})

#### Pending Tool Call

A tool call that is currently running:

In [ ]:
showToolCall({
  toolName: "execute_code",
  input: 'console.log("Hello, world!")',
  status: "pending",
  summary: "Running JavaScript code",
})

#### Error Tool Call

A tool call that encountered an error:

In [ ]:
showToolCall({
  toolName: "read_file",
  input: '{"path": "/nonexistent/file.txt"}',
  status: "error",
  output: "Error: ENOENT: no such file or directory, open '/nonexistent/file.txt'",
})

#### Awaiting Approval

A tool call that requires user approval before executing:

In [ ]:
showToolCall({
  toolName: "delete_file",
  input: '{"path": "/home/user/important_data.csv"}',
  status: "awaiting_approval",
  summary: "Delete a file",
})

#### All Statuses

Here are all the supported tool call statuses displayed together:

In [ ]:
const statuses = ["pending", "awaiting_approval", "approved", "rejected", "completed", "error"];

for (const status of statuses) {
  showToolCall({
    toolName: "example_tool",
    input: JSON.stringify({ action: "demo", status }),
    status,
    summary: `Status: ${status}`,
    ...(["completed", "error", "rejected"].includes(status) && { output: "Some output" }),
  });
}

## InlineDiff Component

The `InlineDiff` component renders one or more file diffs directly in chat output.

**Component name**: `"inline-diff"`

### Metadata Fields

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `diffs` | `Array` | Yes | List of file diffs to render |

Each diff item uses the following shape:

| Field | Type | Required | Description |
|-------|------|----------|-------------|
| `path` | `string` | Yes | Path shown in the diff header |
| `newText` | `string` | Yes | Updated file contents |
| `oldText` | `string` | No | Previous file contents |


In [ ]:
function showInlineDiff(path, oldText, newText) {
  display(
    { _toMime: () => ({ [MIME_TYPE]: "inline-diff" }) },
    { [MIME_TYPE]: { diffs: [{ path, oldText, newText }] } },
  );
}

### Example

Define the original and updated file contents in variables, then render their diff:

In [ ]:
const oldText = `def greet(name):
    return f\"Hello {name}\"`;

const newText = `def greet(name, excited = false):
    suffix = \"!\" if excited else \"\"\n    return f\"Hello {name}{suffix}\"`;

In [ ]:
showInlineDiff("src/example.py", oldText, newText);

### Longer Diff with Truncation

When a diff exceeds 20 lines, the component truncates the output and shows a
clickable "... N more lines" button to expand the remaining lines:

In [ ]:
const longOldText = `import os
import sys


def read_file(path):
    """Read a file and return its contents."""
    with open(path, "r") as f:
        return f.read()


def write_file(path, content):
    """Write content to a file."""
    with open(path, "w") as f:
        f.write(content)


def list_files(directory):
    """List all files in a directory."""
    return os.listdir(directory)


def get_extension(filename):
    """Get the file extension."""
    return os.path.splitext(filename)[1]


def is_python_file(filename):
    """Check if a file is a Python file."""
    return get_extension(filename) == ".py"


def count_lines(path):
    """Count the number of lines in a file."""
    content = read_file(path)
    return len(content.splitlines())`;

const longNewText = `import os
import sys
from pathlib import Path


def read_file(path):
    """Read a file and return its contents."""
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


def write_file(path, content):
    """Write content to a file."""
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)


def list_files(directory, pattern="*"):
    """List all files in a directory matching a pattern."""
    return [str(p) for p in Path(directory).glob(pattern)]


def get_extension(filename):
    """Get the file extension."""
    return Path(filename).suffix


def is_python_file(filename):
    """Check if a file is a Python file."""
    return get_extension(filename) == ".py"


def count_lines(path):
    """Count the number of lines in a file."""
    content = read_file(path)
    return len(content.splitlines())


def find_files(directory, extension):
    """Find all files with a given extension."""
    return [f for f in list_files(directory) if f.endswith(extension)]`;

In [ ]:
showInlineDiff("src/utils.py", longOldText, longNewText);